# Gráfico da Taxa por Ano — Nível Estadual

Para cada variável, agrega Minas Gerais inteiro ano a ano e gera um
gráfico com barras (valor absoluto, eixo esquerdo) e linha (taxa por
100 mil habitantes, eixo direito).




## Imports

In [1]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

from utilitarios_populacional import (
    COLUNA_CODIGO_IBGE,
    COLUNA_ANO,
    COLUNA_TIPO,
    COLUNA_QUANTIDADE,
    COLUNA_POPULACAO,
    COLUNA_TAXA,
    PREFIXO_COLUNA_TOTAL_ANO,
    MULTIPLICADOR_TAXA,
    registrar_log,
    carregar_taxa,
    montar_grafico_barra_linha,
    montar_grafico_empilhado,
    salvar_figura,
    melt_variavel_categorica,
    ordenar_categorias,
    gerar_paleta_cores,
)

## Configuração Geral

In [ ]:
PASTA_TAXAS = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional")
CAMINHO_POPULACAO = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv")

PASTA_SAIDA = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras")

LARGURA_GRAFICO = 1050
ALTURA_GRAFICO = 650

LISTA_VARIAVEIS = [
    {
        "NOME": "Casos",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "casos_taxa_por_100_mil.csv",
        "ANO_INICIAL": 1985,
        "ANO_FINAL": 2023,
        "TITULO_GRAFICO": "Evolução dos Casos de Câncer de Mama em Minas Gerais",
        "NOME_ABSOLUTO": "Casos Absolutos",
        "COR_BARRAS": "#7b2d8e",
        "COR_LINHA": "#2a9d8f",
        "NOME_ARQUIVO_SAIDA": "taxa_casos_por_ano",
    },
    {
        "NOME": "Mortalidade",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "mortalidade_taxa_por_100_mil.csv",
        "ANO_INICIAL": 1979,
        "ANO_FINAL": 2024,
        "TITULO_GRAFICO": "Evolução da Mortalidade por Câncer de Mama em Minas Gerais",
        "NOME_ABSOLUTO": "Óbitos Absolutos",
        "COR_BARRAS": "#c0392b",
        "COR_LINHA": "#2c3e50",
        "NOME_ARQUIVO_SAIDA": "taxa_mortalidade_por_ano",
    },
    {
        "NOME": "Mamografia",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "mamografia_taxa_por_100_mil.csv",
        "ANO_INICIAL": 2013,
        "ANO_FINAL": 2025,
        "TITULO_GRAFICO": "Evolução das Mamografias por Câncer de Mama em Minas Gerais",
        "NOME_ABSOLUTO": "Mamografias Absolutas",
        "COR_BARRAS": "#0e7c7b",
        "COR_LINHA": "#d17a22",
        "NOME_ARQUIVO_SAIDA": "taxa_mamografia_por_ano",
    },
]

CAMINHO_HOSPITAIS_POR_TIPO = PASTA_TAXAS / "hospitais_por_tipo_taxa_por_100_mil.csv"

CONFIGURACAO_HOSPITAIS = {
    "TITULO_GRAFICO": "Hospitais Habilitados por Tipo de Serviço em Minas Gerais (2023)",
    "NOME_ABSOLUTO": "Habilitações Absolutas",
    "COR_BARRAS": "#8a5a44",
    "COR_LINHA": "#1f7a4d",
    "NOME_ARQUIVO_SAIDA": "taxa_hospitais_por_tipo",
}

### A taxa estadual é sempre calculada como **razão das somas**:

$$\text{Taxa}_{\text{ano}} = \frac{\sum_{\text{município}} \text{Quantidade}}{\sum_{\text{município}} \text{População}} \times 100.000$$


In [3]:
def agregar_estadual_por_grupo(tabela_taxa, coluna_de_agrupamento, nome_da_variavel):
    '''
    Agrega quantidade e população somadas de todos os municípios,
    agrupando pela coluna informada (ANO ou TIPO), e calcula a taxa
    como razão das somas (não como média das taxas municipais).

    Antes de agregar, descarta linhas em que POPULACAO é NaN. Sem isso,
    o .sum() por coluna soma o QUANTIDADE dessas linhas normalmente
    (ignorando o NaN) mas ignora a população ausente no denominador —
    ou seja, o caso entra no numerador sem uma população válida por
    trás, inflando a taxa agregada. Com o filtro, o caso só entra na
    soma do grupo se a população daquele município/ano também estiver
    disponível.
    '''
    linhas_sem_populacao = tabela_taxa[COLUNA_POPULACAO].isna()
    if linhas_sem_populacao.any():
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {linhas_sem_populacao.sum()} linha(s) sem "
            f"POPULACAO válida foram excluídas de QUANTIDADE e POPULACAO antes da "
            f"agregação (para não inflar a taxa agregada)."
        )
    tabela_taxa_valida = tabela_taxa[~linhas_sem_populacao]

    agregado = tabela_taxa_valida.groupby(coluna_de_agrupamento)[[COLUNA_QUANTIDADE, COLUNA_POPULACAO]].sum()

    grupos_sem_populacao = agregado[COLUNA_POPULACAO].isna() | (agregado[COLUNA_POPULACAO] == 0)
    if grupos_sem_populacao.any():
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {grupos_sem_populacao.sum()} grupo(s) sem "
            f"população correspondente. A taxa desses grupos ficará como NaN."
        )
    else:
        registrar_log(f"Todos os grupos de {nome_da_variavel} têm população correspondente.")

    agregado[COLUNA_TAXA] = (agregado[COLUNA_QUANTIDADE] / agregado[COLUNA_POPULACAO]) * MULTIPLICADOR_TAXA
    return agregado

## Célula 4 — Processamento e Geração dos Gráficos (Casos, Mortalidade, Mamografia)

Percorre a lista `LISTA_VARIAVEIS`, lendo o CSV de taxa já calculado
de cada variável e agregando por ano.

In [ ]:
dicionario_figuras_por_variavel = {}

# Célula 4 — loop principal: para cada variável, chama carregar_taxa (lê CSV + filtra intervalo de anos) → agregar_estadual_por_grupo agrupando por ANO → ordena por ano → arredonda a taxa (1 casa decimal) → 
# chama montar_grafico_barra_linha passando eixo X (anos), valores absolutos, valores de taxa, título, cores, eixo_x_ano_continuo=True (isso ativa tickmode="linear", dtick=1 no eixo X, garantindo que todo ano apareça, sem pular). 
# A figura é salva em PNG+HTML e guardada em dicionario_figuras_por_variavel.

for configuracao_da_variavel in LISTA_VARIAVEIS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_taxa = carregar_taxa(
        configuracao_da_variavel["CAMINHO_ENTRADA"],
        configuracao_da_variavel["ANO_INICIAL"],
        configuracao_da_variavel["ANO_FINAL"],
    )

    agregado_por_ano = agregar_estadual_por_grupo(tabela_taxa, COLUNA_ANO, nome_da_variavel)
    agregado_por_ano = agregado_por_ano.sort_index()
    agregado_por_ano[COLUNA_TAXA] = agregado_por_ano[COLUNA_TAXA].round(1)

    figura_da_variavel = montar_grafico_barra_linha(
        eixo_x=agregado_por_ano.index.tolist(),
        valores_absolutos=agregado_por_ano[COLUNA_QUANTIDADE].values,
        valores_taxa=agregado_por_ano[COLUNA_TAXA].values,
        titulo=configuracao_da_variavel["TITULO_GRAFICO"],
        nome_absoluto=configuracao_da_variavel["NOME_ABSOLUTO"],
        cor_barras=configuracao_da_variavel["COR_BARRAS"],
        cor_linha=configuracao_da_variavel["COR_LINHA"],
        titulo_eixo_x="Ano",
        largura=LARGURA_GRAFICO,
        altura=ALTURA_GRAFICO,
        eixo_x_ano_continuo=True,
    )

    salvar_figura(figura_da_variavel, configuracao_da_variavel["NOME_ARQUIVO_SAIDA"], PASTA_SAIDA)

    dicionario_figuras_por_variavel[nome_da_variavel] = figura_da_variavel

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} =====\n")

[LOG] ===== INÍCIO: CASOS =====
[LOG] ATENÇÃO (Casos): 780 linha(s) sem POPULACAO válida foram excluídas de QUANTIDADE e POPULACAO antes da agregação (para não inflar a taxa agregada).
[LOG] Todos os grupos de Casos têm população correspondente.


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_casos_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_casos_por_ano.html
[LOG] ===== FIM: CASOS =====

[LOG] ===== INÍCIO: MORTALIDADE =====
[LOG] ATENÇÃO (Mortalidade): 1387 linha(s) sem POPULACAO válida foram excluídas de QUANTIDADE e POPULACAO antes da agregação (para não inflar a taxa agregada).
[LOG] Todos os grupos de Mortalidade têm população correspondente.


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mortalidade_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mortalidade_por_ano.html
[LOG] ===== FIM: MORTALIDADE =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====
[LOG] ATENÇÃO (Mamografia): 78 linha(s) sem POPULACAO válida foram excluídas de QUANTIDADE e POPULACAO antes da agregação (para não inflar a taxa agregada).
[LOG] Todos os grupos de Mamografia têm população correspondente.


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mamografia_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mamografia_por_ano.html
[LOG] ===== FIM: MAMOGRAFIA =====



## Célula 5 — Exibição: Casos

In [5]:
dicionario_figuras_por_variavel["Casos"].show()

## Célula 6 — Exibição: Mortalidade

In [6]:
dicionario_figuras_por_variavel["Mortalidade"].show()

## Célula 7 — Exibição: Mamografia

In [7]:
dicionario_figuras_por_variavel["Mamografia"].show()

---
## Hospitais



In [8]:
tabela_hospitais_por_tipo = carregar_taxa(CAMINHO_HOSPITAIS_POR_TIPO)

quantidade_por_tipo = tabela_hospitais_por_tipo.groupby(COLUNA_TIPO)[COLUNA_QUANTIDADE].sum()

tabela_populacao = pd.read_csv(CAMINHO_POPULACAO)
populacao_total_mg_2023 = tabela_populacao[f"{PREFIXO_COLUNA_TOTAL_ANO}2023"].sum()

taxa_por_tipo = ((quantidade_por_tipo / populacao_total_mg_2023) * MULTIPLICADOR_TAXA).round(1)

tipos_ordenados = quantidade_por_tipo.sort_values(ascending=False).index
quantidade_por_tipo = quantidade_por_tipo.reindex(tipos_ordenados)
taxa_por_tipo = taxa_por_tipo.reindex(tipos_ordenados)

nomes_dos_tipos = [tipo.title() for tipo in quantidade_por_tipo.index]

registrar_log(f"População total de MG em 2023 usada como denominador: {populacao_total_mg_2023:,.0f}")

figura_hospitais = montar_grafico_barra_linha(
    eixo_x=nomes_dos_tipos,
    valores_absolutos=quantidade_por_tipo.values,
    valores_taxa=taxa_por_tipo.values,
    titulo=CONFIGURACAO_HOSPITAIS["TITULO_GRAFICO"],
    nome_absoluto=CONFIGURACAO_HOSPITAIS["NOME_ABSOLUTO"],
    cor_barras=CONFIGURACAO_HOSPITAIS["COR_BARRAS"],
    cor_linha=CONFIGURACAO_HOSPITAIS["COR_LINHA"],
    titulo_eixo_x="Tipo de Hospital",
    largura=LARGURA_GRAFICO,
    altura=ALTURA_GRAFICO,
)

salvar_figura(figura_hospitais, CONFIGURACAO_HOSPITAIS["NOME_ARQUIVO_SAIDA"], PASTA_SAIDA)

dicionario_figuras_por_variavel["Hospitais"] = figura_hospitais

[LOG] População total de MG em 2023 usada como denominador: 10,650,309


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_tipo.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_tipo.html


## Célula 9 — Exibição: Hospitais

In [9]:
dicionario_figuras_por_variavel["Hospitais"].show()

---
# Gráficos de Barras Empilhadas — Categoria, ao Longo dos Anos



## Célula 10 — Configuração dos Gráficos Empilhados

In [ ]:
CATEGORIAS_EXCLUIDAS = ()   # ex.: ("SEM INFORMAÇÃO",) para excluir essa categoria dos gráficos

LARGURA_GRAFICO_EMPILHADO = LARGURA_GRAFICO
ALTURA_GRAFICO_EMPILHADO = ALTURA_GRAFICO

LISTA_VARIAVEIS_CATEGORICAS = [
    {
        "NOME": "Faixa Etária",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _faixa_etaria.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Faixa Etária, ao Longo dos Anos",
        "ORDEM_CATEGORIAS": ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 OU MAIS"],
        "NOME_ARQUIVO_SAIDA": "empilhado_faixa_etaria_por_ano",
    },
    {
        "NOME": "Raça/Cor",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _raca_cor.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Raça/Cor, ao Longo dos Anos",
        "ORDEM_CATEGORIAS": None,   # None = ordem alfabética, com SEM INFORMAÇÃO por último
        "NOME_ARQUIVO_SAIDA": "empilhado_raca_cor_por_ano",
    },
    {
        "NOME": "Escolaridade",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _escolaridade.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Escolaridade, ao Longo dos Anos",
        "ORDEM_CATEGORIAS": [
            "NENHUMA", "FUNDAMENTAL INCOMPLETO", "FUNDAMENTAL COMPLETO", "NÍVEL MÉDIO",
            "NÍVEL SUPERIOR INCOMPLETO", "NÍVEL SUPERIOR COMPLETO", "SEM INFORMAÇÃO",
        ],
        "NOME_ARQUIVO_SAIDA": "empilhado_escolaridade_por_ano",
    },
]

## Processamento e Geração dos Gráficos Empilhados

In [11]:
dicionario_figuras_categoricas = {}

for configuracao_da_variavel in LISTA_VARIAVEIS_CATEGORICAS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO (empilhado): {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_longa = melt_variavel_categorica(tabela_bruta, COLUNA_CODIGO_IBGE)

    if CATEGORIAS_EXCLUIDAS:
        tabela_longa = tabela_longa[~tabela_longa["CATEGORIA"].isin(CATEGORIAS_EXCLUIDAS)]

    tabela_agregada = tabela_longa.groupby(["ANO", "CATEGORIA"], as_index=False)["VALOR"].sum()
    tabela_pivotada = tabela_agregada.pivot(index="ANO", columns="CATEGORIA", values="VALOR").fillna(0)

    ordem_categorias = ordenar_categorias(
        tabela_pivotada.columns.tolist(), configuracao_da_variavel["ORDEM_CATEGORIAS"]
    )
    paleta_cores = gerar_paleta_cores(ordem_categorias)

    figura_da_variavel = montar_grafico_empilhado(
        eixo_x=tabela_pivotada.index.tolist(),
        tabela_pivotada=tabela_pivotada,
        titulo=configuracao_da_variavel["TITULO_GRAFICO"],
        paleta_cores=paleta_cores,
        titulo_eixo_x="Ano",
        ordem_categorias=ordem_categorias,
        largura=LARGURA_GRAFICO_EMPILHADO,
        altura=ALTURA_GRAFICO_EMPILHADO,
    )

    salvar_figura(figura_da_variavel, configuracao_da_variavel["NOME_ARQUIVO_SAIDA"], PASTA_SAIDA)

    dicionario_figuras_categoricas[nome_da_variavel] = figura_da_variavel

    registrar_log(f"===== FIM (empilhado): {nome_da_variavel.upper()} =====\n")

[LOG] ===== INÍCIO (empilhado): FAIXA ETÁRIA =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_faixa_etaria_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_faixa_etaria_por_ano.html
[LOG] ===== FIM (empilhado): FAIXA ETÁRIA =====

[LOG] ===== INÍCIO (empilhado): RAÇA/COR =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_raca_cor_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_raca_cor_por_ano.html
[LOG] ===== FIM (empilhado): RAÇA/COR =====

[LOG] ===== INÍCIO (empilhado): ESCOLARIDADE =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_escolaridade_por_ano.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_escolaridade_por_ano.html
[LOG] ===== FIM (empilhado): ESCOLARIDADE =====



## Exibição: Faixa Etária

In [12]:
dicionario_figuras_categoricas["Faixa Etária"].show()

## Exibição: Raça/Cor

In [13]:
dicionario_figuras_categoricas["Raça/Cor"].show()

## Exibição: Escolaridade

In [14]:
dicionario_figuras_categoricas["Escolaridade"].show()